# Cleveland data exploration

Explore the training partition only. The final test partition is reserved by `heart.data.split_dataset`.
Use the project's Python 3.12 virtual environment as the notebook kernel.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'heart').is_dir():
    root = root.parent
if not (root / 'heart').is_dir():
    raise RuntimeError('Open this notebook from the project or notebooks folder.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from heart.data import load_dataset, split_dataset
from heart.config import NUMERIC

X, y = load_dataset()
X_train, X_test, y_train, y_test = split_dataset(X, y)
print(f'Training rows: {len(X_train)}; reserved test rows: {len(X_test)}')


## Check schema and missing values
Numeric-looking category codes are documented in `docs/DATA_DICTIONARY.md`. Do not impute before cross-validation.

In [ ]:
X_train.head()

In [ ]:
pd.DataFrame({'dtype': X_train.dtypes.astype(str), 'missing': X_train.isna().sum(), 'unique': X_train.nunique()})

In [ ]:
X_train[NUMERIC].describe().T

## Class balance
A majority-class baseline provides context for accuracy.

In [ ]:
counts = y_train.value_counts().sort_index()
display(counts.rename(index={0: 'Absent', 1: 'Present'}).to_frame('count'))
counts.plot.bar(color=['#18786F', '#D68A35'], rot=0, title='Training class counts')
plt.xlabel('Binary target'); plt.ylabel('Records'); plt.show()


## Distributions by class
These plots show associations, not causal effects.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for name, ax in zip(NUMERIC, axes.flat):
    for label in [0, 1]:
        ax.hist(X_train.loc[y_train == label, name].dropna(), bins=12, alpha=0.55, label=str(label))
    ax.set_title(name)
axes[0, 0].legend(title='Target')
plt.tight_layout(); plt.show()


## Numeric correlations
Category identifiers are excluded from this Pearson correlation matrix.

In [ ]:
corr = X_train[NUMERIC].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='BrBG')
ax.set_xticks(range(len(NUMERIC)), NUMERIC, rotation=45, ha='right')
ax.set_yticks(range(len(NUMERIC)), NUMERIC)
for i in range(len(NUMERIC)):
    for j in range(len(NUMERIC)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


## Record your observations

- Which values are missing, and how should they be handled?
- Which measurements appear associated with the target?
- Which relationships might be redundant?
- What does the size of this dataset imply about confidence in the findings?

Write observations in `docs/EXPERIMENT_NOTES.md`. Train using `python -m heart.train` from the project root.
Do not use the test partition above to choose preprocessing, parameters or features.
